# Evaluation of the effect of RAG and Chain-of-Thought on the performance of large language models

* Load the packages and the MMLU datasets

In [ ]:
#This notebook will evaluate different answer generation functions on the mmlu dataset, summarizing the results and outputting them to CSV files.

import pandas as pd
import general_utils
#Import other answer generation functions as needed
from evaluate_model_fast import batch_eval
from rag import rag_pipeline, load_vector_store

# load Gemini client
client = general_utils.load_gemini_client()

# MMLU datasets to evaluate
dataset_names = ['anatomy', 'formal_logic', 'moral_scenarios']

# Define prompts we want to try
prompt_db = {"standard": "Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. ",
             "cot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_max_thinking": "Take your time and think carefully before answering. Use the maximum amount of thinking, don’t rush - I want your best answer. It’s better to think too much than too little. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Provide a summary of your reasoning in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_emotional_manipulation": "Your ability to correctly answer this question is extremely important to my career. I will lose my job if you do not answer correctly. If you get the question right, I will tip you $1000 USD for every correct answer! Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_medical_career": "I'm a medical professional. Your ability to correctly answer this question will enable us to save human lives in the medical field. Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_persona": "You are an expert in \{domain\} with 50 years of working experience. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "rag": "Please only use the provided content to answer the question above. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field.",
             "rag_optional": "You have the option to use the provided additional content to answer the question above. Please feel free to disregard the content if it is not helpful or not relevant. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field."
             }

# Range of k values to try for RAG
k_range = range(1,10, 1)


/Users/skysheng/miniforge3/envs/rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/skysheng/miniforge3/envs/rag/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


## Evaluation of Gemini's performance on MMLU datasets

1. Evaluate Standard Model Performance

We simply prompted Gemini to answer the questions from the MMLU datasets. The prompt we provided is very basic prompt with no prompt engineering or chain-of-thought prompting.

2. Evaluate Chain-of-Thought (COT) Model Performance

We prompted Gemini to answer the questions from the MMLU datasets. The prompt we provided is asking the model to think step-by-step before answering the question. We created various prompt engineering techniques to see which prompt engineering techniques are most effective for Gemini. The techniques are:

* Basic COT asking the model to think step-by-step
* COT asking the model to take its time to think as much as possible
* COT with emotional manipulation (e.g., I will give you $1000 if you get the question right)
* COT hinting that I'm from a medical career (previous work shows model tries extra hard when it thinks the context is for human medical field)
* COT with persona (e.g., you are an expert in {domain} with 50 years of working experience)

3. Evaluate Model Performance with RAG

We converted concept net to RAG vector store and used it as context to answer the questions from the MMLU datasets. We tested different k values for the RAG model, k ranges from 1 to 9 with a step of 1. We also tested 2 variants for the RAG prompt engineering techniques: 

* RAG asking the model to only use the provided content to answer the question
* RAG asking the model to use the provided content to answer the question if it is helpful and relevant, but have the option to disregard the content if it is not helpful or not relevant.

> 💡 Important: I've created the vector store, saved it locally in a folder called "chroma_langchain_db", zipped the folder and uploaded to this [Google Drive link](https://drive.google.com/file/d/12_Yagf9kUyb3b_0hGDyRVrop_ls9lRTg/view?usp=sharing). You can download the file, unzip it and place it under `scripts/chroma_langchain_db`, then continue running the code below to load the vector store locally.

In [2]:
# reload cached vector store. Must be saved under "scripts/chroma_langchain_db" if from the root directory
vector_store = load_vector_store(persist_directory="./chroma_langchain_db")

> ⚠️ Warning: Takes ~50 minutes if you want to create the vector store from scratch by running the code below.

In [4]:
# build the rag pipeline to vectorize concept net
#file_path = "../input_data/concept_net.txt"
#vector_store = rag_pipeline(file_path, split_by="newline", chunk_size=1000, chunk_overlap=0, add_start_index=True, batch_processing_size=700)

Run all the variants in batch below: 

In [3]:
master_results_df = batch_eval(dataset_names,
    prompt_db,
    client,
    k_range,
    debugging=False,
    vector_store=vector_store,
    max_workers=10,
    folder_path="../results")

Batch Evaluation Progress:  27%|██▋       | 13/48 [10:15<27:36, 47.32s/it]Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=019c31f1-2c9d-7941-b042-30b60aeebce8,id=019c31f1-2c9d-7941-b042-30b60aeebce8; trace=019c31f1-2c9d-7941-b042-30b60aeebce8,id=019c31f1-2c9d-7941-b042-30b60aeebce8; trace=019c31f1-2e97-7be3-a355-9f363cb44e97,id=019c31f1-2e97-7be3-a355-9f363cb44e97; trace=019c31f1-2e97-7be3-a355-9f363cb44e97,id=019c31f1-2e97-7be3-a355-9f363cb44e97; trace=019c31f1-301c-74f2-a375-b507c9e722f0,id=019c31f1-301c-74f2-a375-b507c9e722f0; trace=019c31f1-301c-74f2-a375-b507c9e722f0,id=019c31f1-301c-74f2-a375-b507c9e722f0
Failed to send compressed multipart ingest: langsmith

In [19]:
#create a dataframe to summarize the results
results_summary = pd.DataFrame({
    'Dataset': ['Standard', 'CoT', 'RAG'],
    'Anatomy': [accuracy_anatomy_std, accuracy_anatomy_cot, accuracy_anatomy_rag],
    'Formal Logic': [accuracy_formal_logic_std, accuracy_formal_logic_cot, accuracy_formal_logic_rag],
    'Moral Scenarios': [accuracy_moral_scenarios_std, accuracy_moral_scenarios_cot, accuracy_moral_scenarios_rag]
}) 
print(results_summary)

results_summary.to_csv('../results/model_performance_summary.csv', index=False)

    Dataset  Anatomy  Formal Logic  Moral Scenarios
0  Standard     0.92      0.979381             0.83
1       CoT     0.90      0.989247             0.84
2       RAG     0.00      0.000000             0.00


In [ ]:
#Generate a dataframe with 20 randomly selected errors for each model:
def get_error_samples(results_df, model_name):
    errors_df = results_df[results_df['correct'] == False]
    sample_errors_df = errors_df.sample(n=min(20, len(errors_df)), random_state=42)
    sample_errors_df['model'] = model_name
    return sample_errors_df

#Generate a csv file containing at most 20 randomly selected errors for each model
error_samples_std = get_error_samples(results_df_std, 'Standard')
error_samples_cot = get_error_samples(results_df_cot, 'CoT')
error_samples_rag = get_error_samples(results_df_rag, 'RAG')
error_samples_df = pd.concat([error_samples_std, error_samples_cot, error_samples_rag], ignore_index=True)
error_samples_df.to_csv('../results/model_error_samples.csv', index=False)